> **เวอร์ชันอ่านจาก CSV + จับเวลา** (ตัวเดียวกับ `pipeline_main.ipynb` เดิม เพิ่มแค่ `%%time` ใน section 2, 3, 6, 7, 9) ใช้เทียบกับเวอร์ชัน DuckDB (`pipeline_main_duckdb.ipynb`) ด้วยจุดจับเวลาเดียวกัน

# Pipeline หลัก: ไข้เลือดออก × อากาศ (CSS 342 / TRENDS-Thai)

Notebook นี้เป็น **boilerplate กลาง** ของทีม: อ่าน `trends_weather_merged.csv` → สร้าง target **D** และฟีเจอร์ → ตัดแบ่ง fold แบบ rolling-origin → มีตัวช่วยประเมิน → เหลือ section ว่างให้แต่ละคนเพิ่ม preprocessing / โมเดลของตัวเอง

**แนวคิด (ภาพในหัว):** ยืนอยู่ที่สัปดาห์ต้นทาง `t` ของจังหวัดหนึ่ง แล้ววางกล่อง 4 สัปดาห์สองใบติดกัน

```
 t-3  t-2  t-1   t | t+1  t+2  t+3  t+4
 [  R_prev (รู้แล้ว) ] [ R_next (สิ่งที่ทำนาย) ]
 D = log1p(R_next) − log1p(R_prev)        (รายจังหวัด; D>0 = กำลังเร่งขึ้น)
```

**กฎเหล็กของ pipeline นี้**
1. 1 แถว = 1 จังหวัด × 1 สัปดาห์ต้นทาง `t`; ฟีเจอร์ทุกตัวต้องรู้ ณ สิ้นสัปดาห์ `t` เท่านั้น
2. อะไรที่ "เรียนรู้จากข้อมูล" (ค่าปกติ/anomaly ของอากาศ, scaler, ฯลฯ) ต้อง fit จาก **train ของ fold นั้น** เท่านั้น
3. ใช้เฉพาะสัปดาห์เต็ม (`week_days == 7`) และหน้าต่างที่วันที่ต่อเนื่องกันจริง
4. เพิ่มคอลัมน์ใหม่ให้ตั้งชื่อด้วย prefix กลุ่ม (`hist_`, `nat_`, `seas_`, `reg_`, `wx_`, `wxa_`, ของใหม่เช่น `my_`) เพื่อให้เลือกเป็นชุดสำหรับ ablation ได้ง่าย
5. คอลัมน์ที่ขึ้นต้น `diag_` / `label_` ใช้วินิจฉัยหรือตัดแบ่งเท่านั้น **ห้ามใส่เป็นฟีเจอร์**

**วิธีใช้:** รัน section 1-9 ตามลำดับ (ใช้เวลาไม่กี่นาที) แล้วไปเพิ่มงานของตัวเองใน section 10-13

## 1. Config (แก้ที่นี่ที่เดียว)

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

SEED = 342                                   # ใช้ seed เดียวทั้งโปรเจกต์
np.random.seed(SEED)

DATA_PATH = Path("trends_weather_merged.csv")
OUT_DIR = Path("pipeline_outputs")
OUT_DIR.mkdir(exist_ok=True)

H = 4                                        # horizon = 4 สัปดาห์ (ขนาดกล่องด้วย)
TARGET = "D"

# --- ตัวเลือกที่ทีมยังต้องตัดสินใจ (ตั้งค่าเริ่มต้นไว้ก่อน) ---
USE_NATIONAL_FEATURES = True                 # ฟีเจอร์ระดับประเทศ ณ t (ถ้าปิด = ใช้เฉพาะประวัติจังหวัดตัวเอง)
FOURIER_K = 2                                # จำนวนคู่ sin/cos ของฤดูกาล
MAX_WX_LAG = 4                               # lag อากาศ 0..MAX_WX_LAG (ขยายถึง 8 ได้ แต่แถวจะน้อยลงเล็กน้อย)
LOW_COUNT_THRESHOLD = 10                     # แถวที่ผู้ป่วยรวม < 10 ราย ถือเป็นกลุ่มเสียงดัง (ใช้รายงานเท่านั้น)

# อากาศฐาน: ชื่อสั้น -> คอลัมน์ในไฟล์
WX_BASE = {"rain": "rain_wk_mm", "tmax": "tmax_mean_c", "tmin": "tmin_mean_c"}

# --- fold: expanding train, ปีทดสอบ = test_year, gap = H สัปดาห์ (จัดการด้วย label_end) ---
MAIN_FOLDS = [
    {"name": "F1_2019", "train_years": (2016, 2018), "test_year": 2019},
    {"name": "F2_2022", "train_years": (2016, 2021), "test_year": 2022},
    {"name": "F3_2023", "train_years": (2016, 2022), "test_year": 2023},
    {"name": "F4_2024", "train_years": (2016, 2023), "test_year": 2024},
    {"name": "F5_2025", "train_years": (2016, 2024), "test_year": 2025},
]
SUPP_FOLDS = [   # ปีผิดปกติ รายงานแยก ไม่นับในเกณฑ์ H1-H4
    {"name": "S_2020", "train_years": (2016, 2019), "test_year": 2020},
    {"name": "S_2021", "train_years": (2016, 2020), "test_year": 2021},
]

## 2. โหลดข้อมูลและตรวจพื้นฐาน

In [ ]:
%%time
raw = pd.read_csv(DATA_PATH, parse_dates=["Week start", "Week end"])
raw = raw.sort_values(["P-code", "Week start"]).reset_index(drop=True)

print("shape:", raw.shape)

# --- sanity checks (ถ้าข้อไหนไม่ผ่านให้หยุดและดูข้อมูลก่อน) ---
assert raw.shape == (40579, 44), "ขนาดข้อมูลไม่ตรงกับที่ proposal ระบุ"
assert not raw.isna().any().any(), "พบค่าว่าง"
assert not raw.duplicated(["P-code", "Week start"]).any(), "คีย์ (P-code, Week start) ซ้ำ"
assert not raw.duplicated(["P-code", "Year", "Epi week"]).any(), "คีย์ (P-code, Year, Epi week) ซ้ำ"
assert (raw["Dengue total"] == raw["Dengue fever (DF)"] + raw["Dengue haemorrhagic fever (DHF)"]
        + raw["Dengue shock syndrome (DSS)"]).all(), "Dengue total != DF+DHF+DSS"
recomputed = raw["Dengue total"] / raw["Population"] * 1e5
assert np.allclose(recomputed, raw["Dengue rate per 100,000"], atol=0.01), "อัตราต่อแสนไม่ตรงกับที่คำนวณซ้ำ"
assert (raw["rain_wk_mm"] >= 0).all() and (raw["tmax_mean_c"] >= raw["tmin_mean_c"]).all(), "ค่าอากาศผิดปกติ"
assert (raw["weather_days"] == raw["week_days"]).all(), "weather_days != week_days"

print("จังหวัด:", raw["P-code"].nunique(), "| สัปดาห์:", raw["Week start"].nunique(),
      "| ช่วง:", raw["Week start"].min().date(), "→", raw["Week end"].max().date())
print("สัปดาห์สั้น (week_days != 7):", (raw["week_days"] != 7).sum() // raw["P-code"].nunique(), "สัปดาห์")
print("ผ่านทุก check")

## 3. สร้างตารางฐานและฟีเจอร์ (ฟังก์ชันบริสุทธิ์ `build_features`)

ทำเป็นฟังก์ชันเดียวเพื่อให้ทดสอบ leakage ได้ในข้อ 6 (ตัดข้อมูลหลังวันหนึ่งทิ้ง แล้วฟีเจอร์ของแถวก่อนหน้านั้นต้องไม่เปลี่ยน)

รายละเอียดสำคัญ
- **ตัวส่วนประชากร:** ทุกหน้าต่างหารด้วยประชากรของปีที่สัปดาห์ `t` ทั้งหมด จึงไม่เกิดขั้นเทียมที่รอยต่อปี
- **ความต่อเนื่อง:** หน้าต่างต้องห่างกันสัปดาห์ละ 7 วันพอดี (ตัดช่วงที่ขาดวัน 1-2 ม.ค. 2021, 1 ม.ค. 2022 และสัปดาห์สั้นออกอัตโนมัติ)
- **ฟีเจอร์ระดับประเทศ:** ค่าเดียวต่อสัปดาห์ (เหมือนกันทุกจังหวัด) คำนวณจากข้อมูลถึง `t` เท่านั้น

In [ ]:
%%time
def _lag(s, ws, k):
    '''s.shift(k) เฉพาะที่แถวก่อนหน้า k แถวห่างกัน k*7 วันพอดี (ไม่งั้น NaN)'''
    ok = (ws - ws.shift(k)).dt.days.eq(7 * k)
    return s.shift(k).where(ok)


def _province_block(g):
    g = g.sort_values("Week start").copy()
    ws = g["Week start"]
    pop = g["Population"].astype(float)
    full = g["week_days"].eq(7)
    c = g["Dengue total"].where(full).astype(float)           # นับผู้ป่วยเฉพาะสัปดาห์เต็ม

    s4 = c.rolling(4).sum()                                   # ผลรวมสัปดาห์ t-3..t
    ok_b3 = (ws - ws.shift(3)).dt.days.eq(21)
    ok_b7 = (ws - ws.shift(7)).dt.days.eq(49)
    ok_f4 = (ws.shift(-4) - ws).dt.days.eq(28)

    R_prev = s4.where(ok_b3) / pop * 1e5                      # t-3..t
    R_prev2 = s4.shift(4).where(ok_b7) / pop * 1e5            # t-7..t-4
    R_next = s4.shift(-4).where(ok_f4) / pop * 1e5            # t+1..t+4  (เป้าหมาย)

    # ---- target + ตัวช่วยตัดแบ่ง/วินิจฉัย (ห้ามใช้เป็นฟีเจอร์) ----
    g[TARGET] = np.log1p(R_next) - np.log1p(R_prev)
    g["label_end"] = g["Week end"].shift(-4)                  # วันสิ้นสุดของ label (ใช้ตัด gap)
    g["diag_R_prev"] = R_prev
    g["diag_R_next"] = R_next
    g["diag_cases_8w"] = (s4.where(ok_b3) + s4.shift(-4).where(ok_f4))   # ผู้ป่วยรวม t-3..t+4

    # ---- hist_: ประวัติโรคของจังหวัดเอง (รู้ ณ t) ----
    for k in range(4):
        g[f"hist_lograte_lag{k}"] = np.log1p((c if k == 0 else _lag(c, ws, k)) / pop * 1e5)
    g["hist_logR_prev"] = np.log1p(R_prev)
    g["hist_logR_prev2"] = np.log1p(R_prev2)
    g["hist_growth_prev"] = g["hist_logR_prev"] - g["hist_logR_prev2"]

    # ---- wk_ / wx_: อากาศรายสัปดาห์ (เฉพาะสัปดาห์เต็ม) และ lag 0..MAX_WX_LAG ----
    for key, col in WX_BASE.items():
        w = g[col].where(full)
        g[f"wk_{key}"] = w
        for k in range(MAX_WX_LAG + 1):
            g[f"wx_{key}_lag{k}"] = w if k == 0 else _lag(w, ws, k)
    return g


def _national_block(df):
    '''ฟีเจอร์ระดับประเทศ: ใช้ข้อมูลถึง t เท่านั้น (ย้อนหลัง) ค่าเดียวต่อสัปดาห์'''
    n = (df.groupby("Week start")
           .agg(cases=("Dengue total", "sum"), pop=("Population", "sum"),
                wd=("week_days", "min"), n_prov=("P-code", "nunique"))
           .reset_index().sort_values("Week start"))
    assert (n["n_prov"] == df["P-code"].nunique()).all(), "บางสัปดาห์มีจังหวัดไม่ครบ"
    ws = n["Week start"]
    c = n["cases"].where(n["wd"].eq(7)).astype(float)
    s4 = c.rolling(4).sum()
    ok_b3 = (ws - ws.shift(3)).dt.days.eq(21)
    ok_b7 = (ws - ws.shift(7)).dt.days.eq(49)
    R_prev = s4.where(ok_b3) / n["pop"] * 1e5
    R_prev2 = s4.shift(4).where(ok_b7) / n["pop"] * 1e5
    out = pd.DataFrame({"Week start": ws})
    out["nat_lograte_lag0"] = np.log1p(c / n["pop"] * 1e5)
    out["nat_logR_prev"] = np.log1p(R_prev)
    out["nat_growth_prev"] = np.log1p(R_prev) - np.log1p(R_prev2)
    return out


def build_features(raw_df):
    '''ตารางฐาน: target + ฟีเจอร์ที่ไม่ต้อง fit จากข้อมูล (anomaly ทำใน section 7 ต่อ fold)'''
    d = raw_df.sort_values(["P-code", "Week start"]).reset_index(drop=True)
    d = pd.concat([_province_block(g) for _, g in d.groupby("P-code", sort=False)], ignore_index=True)
    d = d.sort_values(["P-code", "Week start"]).reset_index(drop=True)

    d = d.merge(_national_block(d), on="Week start", how="left")

    ang = 2 * np.pi * d["Week start"].dt.dayofyear / 365.25      # seas_: ฤดูกาลแบบเรียบ
    for k in range(1, FOURIER_K + 1):
        d[f"seas_sin{k}"] = np.sin(k * ang)
        d[f"seas_cos{k}"] = np.cos(k * ang)

    reg = pd.get_dummies(d["Health region"].astype(int), prefix="reg").astype(int)  # reg_: ตัวแปรเขต
    d = pd.concat([d, reg], axis=1)
    return d


df = build_features(raw)
print(df.shape)

## 4. ตรวจ target D

In [ ]:
valid_D = df[TARGET].notna()
both = valid_D & df["hist_growth_prev"].notna()
print(f"แถวที่มี D: {valid_D.sum():,} จาก {len(df):,}")
print(f"แถวที่มีทั้ง D และ hist_growth_prev (ต้องย้อนหลัง 8 สัปดาห์): {both.sum():,}   (EDA รอบก่อนรายงาน 32,802 → ต้องเท่ากัน)")
assert both.sum() == 32802, "จำนวนแถวไม่ตรงกับ EDA ให้ตรวจนิยามหน้าต่าง"
print("D สรุป:"); print(df.loc[valid_D, TARGET].describe().round(3))
print("\nแถวที่ผู้ป่วยรวม 8 สัปดาห์ < %d ราย: %.1f%%" %
      (LOW_COUNT_THRESHOLD, 100 * (df.loc[valid_D, "diag_cases_8w"] < LOW_COUNT_THRESHOLD).mean()))
print("\nจำนวนแถวที่มี D ต่อปี (ตามปีของสัปดาห์ t):")
print(df.loc[valid_D].groupby("Year").size().to_frame("n_rows").T)
print("\nค่าเฉลี่ย D ต่อปี (ควรสลับบวก/ลบ ไม่ถูกปีใดครอง):")
print(df.loc[valid_D].groupby("Year")[TARGET].mean().round(3).to_frame("mean_D").T)

# กลุ่มคอลัมน์ฟีเจอร์ (ablation เลือกเป็นชุดจากตรงนี้)
WXA_COLS = [f"wxa_{k}_lag{i}" for k in WX_BASE for i in range(MAX_WX_LAG + 1)]   # สร้างต่อ fold
FEATURE_GROUPS = {
    "hist": [c for c in df.columns if c.startswith("hist_")],
    "nat":  [c for c in df.columns if c.startswith("nat_")],
    "seas": [c for c in df.columns if c.startswith("seas_")],
    "reg":  [c for c in df.columns if c.startswith("reg_")],
    "wx":   [c for c in df.columns if c.startswith("wx_")],
    "wxa":  WXA_COLS,
}
for k, v in FEATURE_GROUPS.items():
    print(f"{k:5s} {len(v):2d} cols  e.g. {v[:3]}")

## 5. ตรวจว่า lag อากาศที่สร้างตรงกับคอลัมน์เดิมในไฟล์

ไฟล์มี `rain_wk_mm_lag1..4` อยู่แล้ว (บล็อก 7 วันก่อน `Week start`) ที่นี่สร้าง lag จาก "สัปดาห์ก่อนหน้า" เอง (เพื่อขยายเกิน lag 4 ได้และทำ anomaly ได้) จึงต้องตรวจว่าสองวิธีให้ค่าเท่ากัน

In [ ]:
checks = []
for key, col in WX_BASE.items():
    for k in range(1, min(MAX_WX_LAG, 4) + 1):
        mine, orig = df[f"wx_{key}_lag{k}"], df[f"{col}_lag{k}"]
        m = mine.notna()
        checks.append({"var": key, "lag": k, "n_compared": int(m.sum()),
                       "max_abs_diff": float((mine[m] - orig[m]).abs().max())})
chk = pd.DataFrame(checks)
print(chk.to_string(index=False))
assert (chk["max_abs_diff"] < 1e-6).all(), "lag ที่สร้างเองไม่ตรงกับคอลัมน์เดิม"
print("lag ตรงกับคอลัมน์เดิมทั้งหมด")

## 6. Leakage test: ตัดข้อมูลหลังวัน T ทิ้งแล้วฟีเจอร์ต้องไม่เปลี่ยน

ถ้าฟีเจอร์ของแถวที่สัปดาห์จบก่อน T เปลี่ยนค่าเมื่อเราลบข้อมูลหลัง T ออก แปลว่ามันแอบใช้อนาคต (target `D` และคอลัมน์ `label_end` / `diag_*` ตั้งใจใช้อนาคต จึงไม่ถูกตรวจ)

In [ ]:
%%time
FEATURE_COLS_BASE = [c for g in ["hist", "nat", "seas", "reg", "wx"] for c in FEATURE_GROUPS[g]]


def leakage_test(cutoff="2022-06-01"):
    T = pd.Timestamp(cutoff)
    trunc = build_features(raw[raw["Week end"] <= T])
    full = df[df["Week end"] <= T].set_index(["P-code", "Week start"])
    part = trunc.set_index(["P-code", "Week start"])
    common = full.index.intersection(part.index)
    bad = {}
    for c in FEATURE_COLS_BASE:
        a, b = full.loc[common, c], part.loc[common, c]
        same = (a.fillna(-9999) - b.fillna(-9999)).abs() < 1e-9
        if not same.all():
            bad[c] = int((~same).sum())
    return bad


for cut in ["2019-03-01", "2022-06-01", "2024-09-01"]:
    bad = leakage_test(cut)
    print(cut, "→", "ผ่าน (ไม่มีฟีเจอร์ใดเปลี่ยน)" if not bad else f"พบ leakage: {bad}")
    assert not bad

## 7. Preprocessing ต่อ fold (anomaly ของอากาศ fit จาก train เท่านั้น)

ลำดับต่อ fold
1. **ค่าปกติอากาศ** = ค่าเฉลี่ย (จังหวัด × เดือน) จากสัปดาห์เต็มที่ **จบก่อนวันเริ่มปีทดสอบ** เท่านั้น
2. anomaly รายสัปดาห์ = ค่าจริง − ค่าปกติ แล้วทำ lag 0..`MAX_WX_LAG` → คอลัมน์ `wxa_*`
3. **train** = ปีใน `train_years` และ `label_end < test_start` (ตัดช่องว่าง H สัปดาห์ ไม่ให้ label ของ train ซ้อนช่วงทดสอบ)
4. **test** = แถวที่ปีของสัปดาห์ต้นทาง = `test_year`
5. ทุกชุดฟีเจอร์ประเมินบน **แถวชุดเดียวกัน** (ตัดแถวที่ฟีเจอร์ใดก็ตามเป็น NaN) เพื่อให้เทียบกันได้ตรงๆ

In [ ]:
%%time
def make_climatology(d, test_start):
    m = d["week_days"].eq(7) & (d["Week end"] < test_start)
    cols = [f"wk_{k}" for k in WX_BASE]
    return d.loc[m].groupby(["P-code", "Month"])[cols].mean().add_suffix("_clim").reset_index()


def add_weather_anomaly(d, clim):
    d = d.merge(clim, on=["P-code", "Month"], how="left")
    for k in WX_BASE:
        d[f"anom_{k}"] = d[f"wk_{k}"] - d[f"wk_{k}_clim"]
    pieces = []
    for _, g in d.groupby("P-code", sort=False):
        g = g.sort_values("Week start").copy()
        ws = g["Week start"]
        for k in WX_BASE:
            for i in range(MAX_WX_LAG + 1):
                g[f"wxa_{k}_lag{i}"] = g[f"anom_{k}"] if i == 0 else _lag(g[f"anom_{k}"], ws, i)
        pieces.append(g)
    return pd.concat(pieces).sort_values(["P-code", "Week start"]).reset_index(drop=True)


ALL_FEATURES = [c for g, cols in FEATURE_GROUPS.items() for c in cols
                if (g != "nat" or USE_NATIONAL_FEATURES)]
_FOLD_CACHE = {}


def get_fold(fold):
    '''คืน dict(train=DataFrame, test=DataFrame) พร้อมใช้; ผลถูก cache ไว้'''
    if fold["name"] in _FOLD_CACHE:
        return _FOLD_CACHE[fold["name"]]
    test_start = pd.Timestamp(f"{fold['test_year']}-01-01")
    d = add_weather_anomaly(df, make_climatology(df, test_start))

    usable = d[TARGET].notna() & d[ALL_FEATURES].notna().all(axis=1)
    is_train = usable & d["Year"].between(*fold["train_years"]) & (d["label_end"] < test_start)
    is_test = usable & d["Year"].eq(fold["test_year"])
    tr, te = d[is_train].reset_index(drop=True), d[is_test].reset_index(drop=True)

    # --- unit test ของ fold (หยุดทันทีถ้าไม่ผ่าน) ---
    assert tr["label_end"].max() < test_start, "label ของ train ล้ำเข้าช่วงทดสอบ"
    assert te["Week start"].min() >= test_start, "test เริ่มก่อนต้นปีทดสอบ"
    assert set(tr["P-code"]) <= set(te["P-code"]) | set(tr["P-code"]), "ตรวจจังหวัด"
    out = {"train": tr, "test": te, "test_start": test_start}
    _FOLD_CACHE[fold["name"]] = out
    return out


summary = []
for f in MAIN_FOLDS + SUPP_FOLDS:
    fd = get_fold(f)
    summary.append({"fold": f["name"], "train_years": f"{f['train_years'][0]}-{f['train_years'][1]}",
                    "test_year": f["test_year"], "n_train": len(fd["train"]), "n_test": len(fd["test"]),
                    "last_train_label_end": fd["train"]["label_end"].max().date()})
pd.DataFrame(summary)

## 8. ตัวช่วยประเมิน (metric ตาม proposal: MAE ของ D และ skill เทียบ baseline D = 0)

- `score_predictions` คืน MAE, skill, RMSE, Spearman, MAE แยกแถว D>0 / D<0 และกลุ่มผู้ป่วยน้อย
- `run_experiment` รับ "โรงงานโมเดล" (ฟังก์ชันที่คืนโมเดลใหม่) + รายชื่อคอลัมน์ฟีเจอร์ แล้ววนทุก fold
- `compare_experiments` เทียบผล 2 การทดลองต่อ fold (ใช้ตอบ H4: ลด MAE ≥ 5% และดีกว่า ≥ 4 จาก 5 fold)

ค่า CI แบบ block bootstrap ยังไม่ได้ใส่ไว้ (ให้เป็นงานใน section 12) แถวในสัปดาห์เดียวกันสัมพันธ์กันสูง **ห้ามใช้ SE แบบถือว่าแถวอิสระ**

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin


def score_predictions(te, pred):
    y = te[TARGET].to_numpy()
    err = np.abs(y - pred)
    mae, mae0 = err.mean(), np.abs(y).mean()
    low = (te["diag_cases_8w"] < LOW_COUNT_THRESHOLD).to_numpy()
    return {
        "n": len(y), "mae": mae, "mae_zero": mae0, "skill": 1 - mae / mae0,
        "rmse": float(np.sqrt(np.mean((y - pred) ** 2))),
        "spearman": float(pd.Series(y).rank().corr(pd.Series(pred).rank())) if np.std(pred) > 0 else np.nan,
        "mae_D_pos": err[y > 0].mean() if (y > 0).any() else np.nan,
        "mae_D_neg": err[y < 0].mean() if (y < 0).any() else np.nan,
        "mae_low_cases": err[low].mean() if low.any() else np.nan,
        "mae_high_cases": err[~low].mean() if (~low).any() else np.nan,
    }


def run_experiment(name, model_factory, feature_cols, folds=MAIN_FOLDS, verbose=True):
    '''คืน (ตารางผลต่อ fold, ตาราง prediction ต่อแถว)'''
    rows, preds = [], []
    for f in folds:
        fd = get_fold(f)
        tr, te = fd["train"], fd["test"]
        model = model_factory()
        model.fit(tr[feature_cols], tr[TARGET])
        p = np.asarray(model.predict(te[feature_cols]), dtype=float)
        rows.append({"experiment": name, "fold": f["name"], "test_year": f["test_year"], **score_predictions(te, p)})
        preds.append(te[["P-code", "Week start", "Year", "Health region", TARGET, "diag_cases_8w"]]
                     .assign(pred=p, fold=f["name"], experiment=name))
    res = pd.DataFrame(rows)
    if verbose:
        print(f"[{name}] mean MAE = {res['mae'].mean():.4f} | mean skill = {res['skill'].mean():.3f}")
    return res, pd.concat(preds, ignore_index=True)


def compare_experiments(res_base, res_new):
    '''เทียบ MAE ต่อ fold: pct_reduction > 0 แปลว่า res_new ดีกว่า res_base'''
    m = res_base[["fold", "test_year", "mae"]].merge(res_new[["fold", "mae"]], on="fold", suffixes=("_base", "_new"))
    m["pct_reduction"] = 100 * (1 - m["mae_new"] / m["mae_base"])
    m["new_better"] = m["mae_new"] < m["mae_base"]
    print(f"เฉลี่ยลด MAE {m['pct_reduction'].mean():.2f}% | ดีกว่าใน {int(m['new_better'].sum())}/{len(m)} fold")
    return m


class ConstantBaseline(BaseEstimator, RegressorMixin):
    '''ทายค่าคงที่ (c=0 คือ baseline "ไม่เปลี่ยนแปลง")'''
    def __init__(self, c=0.0):
        self.c = c
    def fit(self, X, y=None):
        return self
    def predict(self, X):
        return np.full(len(X), self.c)


class ColumnBaseline(BaseEstimator, RegressorMixin):
    '''ทาย D = ค่าของคอลัมน์หนึ่ง (เช่น hist_growth_prev)'''
    def __init__(self, col="hist_growth_prev"):
        self.col = col
    def fit(self, X, y=None):
        return self
    def predict(self, X):
        return X[self.col].to_numpy()

## 9. Baseline และตัวอย่างโมเดล (เอาไว้ดูรูปแบบการใช้ ไม่ใช่ผลสรุป)

- **B1:** D = 0 และ D = `hist_growth_prev`
- ตัวอย่าง Ridge และ HistGradientBoosting บนชุด **F1 (ประวัติ + ฤดูกาล + เขต)** เท่านั้น ยังไม่ได้ใส่อากาศ ผลตรงนี้จึง **ไม่ใช่คำตอบของ H4**

In [ ]:
%%time
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

# ชุดฟีเจอร์สำหรับ ablation E4 (F2-F4 เติมอากาศบน F1)
G = FEATURE_GROUPS
F1 = G["hist"] + (G["nat"] if USE_NATIONAL_FEATURES else []) + G["seas"] + G["reg"]
FEATURE_SETS = {
    "F1_history_season": F1,
    "F2_plus_wx_raw": F1 + G["wx"],
    "F3_plus_wx_anom": F1 + G["wxa"],
    "F4_plus_both": F1 + G["wx"] + G["wxa"],
}

all_res, all_pred = [], []


def keep(res_pred):
    all_res.append(res_pred[0]); all_pred.append(res_pred[1]); return res_pred


r_zero = keep(run_experiment("B1_zero", lambda: ConstantBaseline(0.0), F1))
r_gprev = keep(run_experiment("B1_growth_prev", lambda: ColumnBaseline("hist_growth_prev"), F1))
r_ridge = keep(run_experiment("M0_ridge_F1", lambda: make_pipeline(StandardScaler(), Ridge(alpha=10.0)), FEATURE_SETS["F1_history_season"]))
r_hgb = keep(run_experiment("M0_hgb_F1",
                            lambda: HistGradientBoostingRegressor(loss="absolute_error", learning_rate=0.05,
                                                                  max_iter=200, random_state=SEED),
                            FEATURE_SETS["F1_history_season"]))
results = pd.concat(all_res, ignore_index=True)
results.pivot(index="fold", columns="experiment", values="mae").round(4)

## 10. [ว่าง] Data preprocessing / transform เพิ่มเติม

ใส่ฟีเจอร์หรือการแปลงของคุณที่นี่ ข้อควรจำ
- ฟีเจอร์ใหม่ที่ **ไม่ต้อง fit** (เช่น ค่าเฉลี่ยเคลื่อนที่ของอากาศ, interaction): สร้างเพิ่มบน `df` โดยดึงจากแถว `t` และอดีตเท่านั้น ตั้งชื่อ `my_...` แล้วใส่เข้า `FEATURE_GROUPS["my"]`
- ฟีเจอร์ที่ **ต้อง fit** (scaler, ค่าเฉลี่ยจังหวัด, PCA ฯลฯ): ทำภายใน pipeline ของโมเดล หรือคำนวณจาก `fd["train"]` ของแต่ละ fold เท่านั้น
- เพิ่มแล้วให้รัน `leakage_test()` ใหม่ (ใส่ชื่อคอลัมน์ใหม่ใน `FEATURE_COLS_BASE` ถ้าเป็นฟีเจอร์ที่ไม่ต้อง fit)
- ถ้าเปลี่ยน `df` ให้ล้าง cache ด้วย `_FOLD_CACHE.clear()` ก่อนเรียก `get_fold`

```python
# ตัวอย่างโครง (ยังไม่ได้รัน): ค่าเฉลี่ยเคลื่อนที่ของฝน 3 สัปดาห์
# df["my_rain_ma3"] = df[[f"wx_rain_lag{k}" for k in range(3)]].mean(axis=1)
# FEATURE_GROUPS["my"] = ["my_rain_ma3"]
# _FOLD_CACHE.clear()
```

In [ ]:
# --- พื้นที่ทำงานของคุณ: preprocessing / feature engineering ---

## 11. [ว่าง] โมเดลที่อยากลอง

รูปแบบเดียวกับ section 9: เขียน "โรงงานโมเดล" (lambda ที่คืนโมเดลใหม่ซึ่งมี `.fit(X, y)` / `.predict(X)`) แล้วเรียก `run_experiment(ชื่อ, โรงงาน, คอลัมน์ฟีเจอร์)` ตั้งชื่อการทดลองให้ไม่ซ้ำ
ใช้ `random_state=SEED` ทุกครั้ง และเทียบกับ baseline ด้วย `compare_experiments`

```python
# ตัวอย่างโครง (ยังไม่ได้รัน)
# r = keep(run_experiment("M?_ชื่อ", lambda: YourModel(random_state=SEED), FEATURE_SETS["F1_history_season"]))
# compare_experiments(r_ridge[0], r[0])
```

In [ ]:
# --- พื้นที่ทำงานของคุณ: โมเดล ---

## 12. [ว่าง] Ablation E4 (ตอบ H4) และการประเมินทางสถิติ

โครงที่ตกลงกันไว้: เทียบ F2 / F3 / F4 กับ **F1** (ไม่ใช่เทียบประวัติเปล่า) ด้วยโมเดลและ hyperparameter เดียวกัน เกณฑ์ H4: ลด MAE ≥ 5%, ดีกว่าใน ≥ 4 จาก 5 fold และ CI ของผลต่างไม่คร่อมศูนย์

ที่ยังต้องเขียน: paired block bootstrap (บล็อกตามปี/สัปดาห์ ไม่ใช่รายแถว) ตารางสรุปผลข้อ 9.3 และ sensitivity E6 (ตัด 2021 / 2020-21) ใช้ `SUPP_FOLDS` และการตัดปีออกจาก train ผ่านการแก้ `train_years`/mask ใน `get_fold`

```python
# ตัวอย่างโครง (ยังไม่ได้รัน)
# mk = lambda: make_pipeline(StandardScaler(), Ridge(alpha=10.0))
# base = run_experiment("E4_F1", mk, FEATURE_SETS["F1_history_season"])
# for k in ["F2_plus_wx_raw", "F3_plus_wx_anom", "F4_plus_both"]:
#     new = run_experiment(f"E4_{k}", mk, FEATURE_SETS[k])
#     compare_experiments(base[0], new[0])
```

In [ ]:
# --- พื้นที่ทำงานของคุณ: ablation / bootstrap ---

## 13. [ว่าง] Error analysis

แยกผลตามปี (โดยเฉพาะ 2021 และ 2023), เขตสุขภาพ, ช่วงพีค/นอกพีค, จังหวัดที่ผู้ป่วยน้อย และรายงาน MAE แยกแถว D>0 / D<0 ตารางที่ `run_experiment` คืนมา (`all_pred`) มีคอลัมน์ `Health region`, `diag_cases_8w`, `pred`, `D` ครบแล้ว

```python
# ตัวอย่างโครง (ยังไม่ได้รัน)
# preds = pd.concat(all_pred, ignore_index=True)
# preds.assign(abs_err=(preds["D"] - preds["pred"]).abs()).groupby(["experiment", "Health region"])["abs_err"].mean().unstack(0)
```

In [ ]:
# --- พื้นที่ทำงานของคุณ: error analysis ---

## 14. บันทึกตารางฐาน (ไม่บังคับ)

เซฟตารางฐาน (ไม่รวม anomaly ที่ต้อง fit ต่อ fold) เพื่อแชร์ในทีมหรือใช้นอก notebook

In [ ]:
# df.to_csv(OUT_DIR / "feature_table_base.csv", index=False)
# print("saved", OUT_DIR / "feature_table_base.csv")